# 03 - Agent Demo

This notebook demonstrates the LangGraph ReAct agent that uses amendment-aware LlamaIndex RAG
to answer questions about indexed contracts.

## Architecture
```
User Query -> LangGraph Agent -> search_contracts tool -> LlamaIndex Query Engine -> Pinecone
                    ^                                           |
                    |___________ synthesized answer ____________|
```

The query engine is **amendment-aware**: when terms conflict between a base contract
and its amendments, it prioritizes the latest version (highest version / latest effective_date).

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first.

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root (needed when running from notebooks/ directory)
load_dotenv(Path('../.env'))

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## Build the Agent

The agent consists of:
- **Azure OpenAI LLM** with tool-calling capability
- **`search_contracts`** tool wrapping the LlamaIndex query engine
- **LangGraph** state machine with ReAct routing (agent -> tools -> agent -> END)

In [ ]:
from contract_lens.agent.graph import build_agent

agent, callback_handler = build_agent(settings)
config = {'callbacks': [callback_handler]} if callback_handler else {}

print('Agent ready.')
print(f'LangFuse tracing: {"enabled" if callback_handler else "disabled"}')

## Helper Function

A utility to invoke the agent and display the full message trace.

In [ ]:
from langchain_core.messages import HumanMessage


def ask(question: str) -> str:
    """Send a question to the agent and print the conversation trace."""
    print(f'User: {question}')
    print('-' * 60)

    result = agent.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config,
    )

    for msg in result['messages']:
        role = msg.__class__.__name__.replace('Message', '')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{role}] Tool call: {tc["name"]}({tc["args"]})')
        elif role == 'Tool':
            preview = msg.content[:200] + '...' if len(msg.content) > 200 else msg.content
            print(f'  [{role}] {preview}')
        elif role != 'Human':
            print(f'  [{role}] {msg.content}')

    print('-' * 60)
    answer = result['messages'][-1].content
    return answer

## Example Queries

### Query 1: Current terms (should return latest amendment, not base)

In [ ]:
ask('What is the current hourly rate for a Senior Architect in the IT service agreement?');

### Query 2: Amendment history

In [ ]:
ask('Show me the history of pricing changes in the IT service agreement ITSVC-001.');

### Query 3: SLA targets (should return amended values)

In [ ]:
ask('What is the P1 incident response time in the Cloud SLA?');

### Query 4: Polish language — lease amendment

In [ ]:
ask('Jaki jest aktualny czynsz najmu biura i co zmienilo sie w aneksie?');

### Query 5: Cross-contract comparison

In [ ]:
ask('Which contracts have been amended and what were the key changes?');

## Summary

The agent demonstrates:
- **Amendment awareness**: Returns the latest effective terms, not outdated base contract values
- **Version history**: Can list chronological changes across contract versions
- **Tool-calling with filters**: Can filter by `contract_id`, `language`, `source_type`
- **Multilingual support**: Works with both English and Polish agreements
- **Observability**: All traces are captured in LangFuse (if configured)

For interactive use, run `python scripts/chat.py` from the terminal.